# QCS M5200 Digitizer Trigger Test

## Setup

In [2]:
import keysight_ktm5200x
import numpy as np
from time import sleep
import matplotlib.pyplot as plt
from datetime import timedelta

m5200_visa_addr = "PXI0::12-0.0::INSTR"

idQuery = False
reset   = False
options = "Simulate=False"

digitizer = keysight_ktm5200x.KtM5200x(
    m5200_visa_addr,
    idQuery,
    reset,
    options
)
print("Digitizer Initialized")

print(" identifier: ",  digitizer.identity.identifier)
print(" revision: ",    digitizer.identity.revision)
print(" vendor: ",      digitizer.identity.vendor)
print(" description:",  digitizer.identity.description)
print(" model: ",       digitizer.identity.instrument_model)
print(" resource: ",    digitizer.driver_operation.io_resource_descriptor)
print(" options: ",     digitizer.system.options)
print(" simulate: ",    digitizer.driver_operation.simulate)

print("Digitizer Connection is completed")

number_of_samples = 4800

digitizer.channels[0].set_daq_config(
    1,
    number_of_samples,
    keysight_ktm5200x.TriggerMode.HW_DIG_TRIG,
    0
)
digitizer.channels[0].external_trigger_config(
    keysight_ktm5200x.TriggerSource.SMB_TRIG1,
    keysight_ktm5200x.ExternalTriggerMode.RISING_EDGE,
    keysight_ktm5200x.SyncMode.IMMEDIATE
)
print("External trigger is set")

digitizer.channels[0].daq_flush()
print("DAQ is flushed")
print(f"Current DAQ data : {digitizer.channels[0].daq_counter}")
print("M5200A is waiting for trigger...")
digitizer.channels[0].daq_start()
print("Waiting for waveform to be captured...")
while True:
    if not (digitizer.channels[0].daq_counter == number_of_samples):
        sleep(1)
        print(f"currently read {digitizer.channels[0].daq_counter}...")
    else:
        print("Waveform is captured")
        break
buffer = np.array([], dtype=np.int16) 
digitizer.channels[0].fetch_waveform_int16(
    buffer,
    number_of_samples
)
plt.clf()
plt.plot(buffer)
plt.show()

print()
while True:
    result = digitizer.utility.error_query()
    print('ErrorQuery: {0}, {1}'.format(result[0], result[1]))
    if result[0] == 0:
        break
digitizer.close()
print('Done!')


ImportError: DLL load failed while importing keysight_ktm5200x: A dynamic link library (DLL) initialization routine failed.